# Predictor Lúdico Mundial 2026

Proyecto demostrativo para portafolio de Data Science.

**Idea:** construir un predictor simple de partidos de fútbol usando ponderaciones heurísticas, resultados parciales y un ajuste de intuición del usuario.

> Este notebook no pretende ser un modelo estadístico oficial ni una recomendación de apuestas. Es un ejercicio lúdico para mostrar estructura de datos, reglas de negocio, evaluación de predicciones y una app interactiva.

## 1. Objetivo

Construir un sistema simple que permita:

1. Registrar predicciones de partidos.
2. Compararlas contra resultados reales.
3. Calcular efectividad en ganador y marcador exacto.
4. Usar una tabla de fuerza relativa por selección.
5. Permitir que el usuario ajuste los porcentajes con su intuición.
6. Proyectar eliminatorias y simular campeón, segundo, tercero y cuarto.

## 2. ¿De dónde salen los porcentajes?

Los porcentajes usados son una **ponderación heurística**. No nacen de un modelo entrenado con miles de partidos, sino de una regla aproximada basada en:

- Jerarquía futbolística histórica.
- Percepción de fortaleza relativa.
- Rendimiento reciente observado en la fase de grupos dentro de este ejercicio.
- Diferencia estimada entre selecciones.
- Ajuste manual o intuición del usuario.

En una versión avanzada se puede reemplazar por datos reales: ranking FIFA, Elo, goles a favor, goles en contra, tiros, xG, localía, lesiones y resultados históricos.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

if Path("../data").exists():
    DATA_DIR = Path("../data")
elif Path("data").exists():
    DATA_DIR = Path("data")
else:
    DATA_DIR = Path(".")

print("Ruta actual:", Path.cwd())
print("Ruta de datos:", DATA_DIR.resolve())

Ruta actual: C:\Users\amena\DESAFIOLATAM\Portafolio_Data_Science\proyecto_10_predictor_mundial_2026\notebooks
Ruta de datos: C:\Users\amena\DESAFIOLATAM\Portafolio_Data_Science\proyecto_10_predictor_mundial_2026\data


## 3. Cargar datos

In [2]:
pred = pd.read_csv(DATA_DIR / "data_predicciones_fase_grupos.csv")
real = pd.read_csv(DATA_DIR / "data_resultados_reales_parciales.csv")
fuerza = pd.read_csv(DATA_DIR / "data_fuerza_equipos.csv")
elim = pd.read_csv(DATA_DIR / "data_proyeccion_eliminatoria.csv")

print("Datos cargados correctamente")

Datos cargados correctamente


## 4. Evaluar efectividad parcial

In [3]:
def ganador(goles_local, goles_visita, local, visita):
    if goles_local > goles_visita:
        return local
    if goles_visita > goles_local:
        return visita
    return 'Empate'

pred_eval = pred.copy()
real_eval = real.copy()
pred_eval['key'] = pred_eval.apply(lambda r: '|'.join(sorted([r['equipo_local'], r['equipo_visita']])), axis=1)
real_eval['key'] = real_eval.apply(lambda r: '|'.join(sorted([r['equipo_local'], r['equipo_visita']])), axis=1)

df = real_eval.merge(pred_eval[['key','equipo_local','equipo_visita','pred_local','pred_visita']], on='key', suffixes=('_real','_pred'))

def alinear_prediccion(row):
    if row['equipo_local_real'] == row['equipo_local_pred']:
        return pd.Series([row['pred_local'], row['pred_visita']])
    return pd.Series([row['pred_visita'], row['pred_local']])

df[['pred_local_alineado','pred_visita_alineado']] = df.apply(alinear_prediccion, axis=1)
df['ganador_real'] = df.apply(lambda r: ganador(r['real_local'], r['real_visita'], r['equipo_local_real'], r['equipo_visita_real']), axis=1)
df['ganador_predicho'] = df.apply(lambda r: ganador(r['pred_local_alineado'], r['pred_visita_alineado'], r['equipo_local_real'], r['equipo_visita_real']), axis=1)
df['acierto_ganador'] = df['ganador_real'] == df['ganador_predicho']
df['acierto_marcador_exacto'] = (df['real_local'] == df['pred_local_alineado']) & (df['real_visita'] == df['pred_visita_alineado'])
df['acierto_total_goles'] = (df['real_local'] + df['real_visita']) == (df['pred_local_alineado'] + df['pred_visita_alineado'])

resumen = pd.DataFrame({
    'métrica': ['Acierto ganador/resultado', 'Acierto marcador exacto', 'Acierto total goles'],
    'aciertos': [df['acierto_ganador'].sum(), df['acierto_marcador_exacto'].sum(), df['acierto_total_goles'].sum()],
    'partidos': [len(df), len(df), len(df)]
})
resumen['porcentaje'] = (resumen['aciertos'] / resumen['partidos'] * 100).round(1)
resumen

,métrica,aciertos,partidos,porcentaje
0,Acierto ganador/resultado,14,19,73.7
1,Acierto marcador exacto,4,19,21.1
2,Acierto total goles,5,19,26.3


In [4]:
df[['grupo','equipo_local_real','equipo_visita_real','pred_local_alineado','pred_visita_alineado','real_local','real_visita','ganador_predicho','ganador_real','acierto_ganador','acierto_marcador_exacto']]

KeyError: "['grupo_real'] not in index"

## 5. Visualización

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(resumen['métrica'], resumen['porcentaje'])
plt.title('Efectividad parcial del predictor lúdico')
plt.ylabel('Porcentaje de acierto')
plt.xticks(rotation=25, ha='right')
plt.ylim(0,100)
plt.show()

## 6. Predictor con ajuste de intuición

In [ ]:
strength_dict = dict(zip(fuerza['equipo'], fuerza['fuerza_base']))

def probabilidad_desde_fuerza(f1, f2):
    diff = f1 - f2
    p1 = 1 / (1 + np.exp(-diff / 10))
    p2 = 1 - p1
    return round(p1 * 100, 1), round(p2 * 100, 1)

def marcador_desde_probabilidad(p1, p2):
    diff = abs(p1 - p2)
    if diff < 8:
        return 1, 1
    elif diff < 18:
        return (2, 1) if p1 > p2 else (1, 2)
    elif diff < 30:
        return (2, 0) if p1 > p2 else (0, 2)
    elif diff < 45:
        return (3, 0) if p1 > p2 else (0, 3)
    else:
        return (4, 0) if p1 > p2 else (0, 4)

def predecir_partido(equipo1, equipo2, ajuste1=0, ajuste2=0):
    f1 = strength_dict.get(equipo1, 65) + ajuste1
    f2 = strength_dict.get(equipo2, 65) + ajuste2
    p1, p2 = probabilidad_desde_fuerza(f1, f2)
    g1, g2 = marcador_desde_probabilidad(p1, p2)
    return {'equipo_1':equipo1,'equipo_2':equipo2,'fuerza_1':f1,'fuerza_2':f2,'prob_1':p1,'prob_2':p2,'pronóstico':f'{equipo1} {g1} - {g2} {equipo2}'}

predecir_partido('Brasil','Japón',ajuste1=0,ajuste2=4)

## 7. Eliminatorias cargadas

In [ ]:
elim

## 8. Simular campeón, segundo, tercero y cuarto

In [ ]:
def ganador_predicho(equipo1, equipo2, ajuste1=0, ajuste2=0):
    prediccion = predecir_partido(equipo1, equipo2, ajuste1, ajuste2)
    if prediccion['prob_1'] >= prediccion['prob_2']:
        return equipo1
    return equipo2

semifinales = [('Brasil','Canadá'), ('Alemania','Países Bajos')]
ganadores = [ganador_predicho(a,b) for a,b in semifinales]
perdedores = [b if ganador_predicho(a,b) == a else a for a,b in semifinales]
campeon = ganador_predicho(ganadores[0], ganadores[1])
segundo = ganadores[1] if campeon == ganadores[0] else ganadores[0]
tercero = ganador_predicho(perdedores[0], perdedores[1])
cuarto = perdedores[1] if tercero == perdedores[0] else perdedores[0]

pd.DataFrame({'posición':['Campeón','Segundo','Tercero','Cuarto'], 'equipo':[campeon,segundo,tercero,cuarto]})

## 9. Próximos pasos

1. Revisar datos base y ajustar fuerzas.
2. Crear gráficos más bonitos para LinkedIn.
3. Ejecutar la app Streamlit.
4. Subir a GitHub como `proyecto_07_predictor_ludico_mundial_2026`.
5. Agregar capturas de pantalla y explicación en README.